# 12 — Architecture, Ports and Adapters, Dependency Injection

Separate domain rules from storage, network, and presentation layers. Learn why explicit dependencies, small interfaces, and transactional boundaries make code easier to test and evolve.

*Advanced · 90–150 min*

## You will learn

- Separate pure domain code from I/O adapters
- Create small protocol-based ports
- Inject repositories and time sources
- Identify boundaries and domain-specific errors

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "advanced", "notebooks"))
    sys.path.insert(0, os.getcwd())

## Examples

### 1. Core domain rules

Domain logic should receive plain input and return plain results without opening files or making requests. The example models a reservation policy, allowing tests to explore conflicts without depending on a database.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Reservation:
    item_id: str
    user_id: str

def can_reserve(item_id, active):
    return item_id not in {reservation.item_id for reservation in active}

reservations = [Reservation("A", "u1")]
print(can_reserve("B", reservations), can_reserve("A", reservations))

**Design review:** Should a production reservation service scan all active reservations every time?

### 2. Ports and adapters

A port specifies required operations while adapters implement storage, network, or in-memory behavior. Protocol typing helps communicate the contract; the interpreter does not enforce the annotation automatically.

In [ ]:
from typing import Protocol

class ReservationRepo(Protocol):
    def exists(self, item_id: str) -> bool: ...
    def save(self, record: Reservation) -> None: ...

class InMemoryRepo:
    def __init__(self):
        self.data = {}
    def exists(self, item_id):
        return item_id in self.data
    def save(self, record):
        self.data[record.item_id] = record

def reserve(repo: ReservationRepo, record: Reservation):
    if repo.exists(record.item_id):
        raise ValueError("already reserved")
    repo.save(record)
    return record

repo = InMemoryRepo()
print(reserve(repo, Reservation("B", "u2")))

**Design review:** Why is the check-then-save implementation not atomic across concurrent callers?

### 3. Injecting time and randomness

Functions that read the real clock or random source can be hard to test. Inject a clock or random generator so tests control nondeterministic values and are not sensitive to wall-clock timing.

In [ ]:
from datetime import date

def membership_expired(expiry, today_fn):
    return expiry < today_fn()

fixed_today = lambda: date(2026, 11, 20)
print(membership_expired(date(2026, 11, 19), fixed_today))
print(membership_expired(date(2026, 11, 20), fixed_today))

**Design review:** Why should a real clock be timezone-aware when working with instants?

### 4. Errors at boundaries

Translate low-level I/O failures at the adapter boundary without hiding their cause. Domain code can raise meaningful exceptions that callers convert into HTTP statuses or CLI error messages.

In [ ]:
class ReservationConflict(Exception):
    pass

def reserve_or_conflict(repo, record):
    try:
        return reserve(repo, record)
    except ValueError as exc:
        raise ReservationConflict("item unavailable") from exc

try:
    reserve_or_conflict(repo, Reservation("B", "u3"))
except ReservationConflict as exc:
    print(type(exc).__name__, exc)

**Design review:** What kind of database constraint is needed to enforce uniqueness under concurrency?

## Try it

Short fill-in exercises on this lesson: replace each `___`, run the cell, and `check` tells you whether you're right. The answer itself stays hidden. More practice for this topic is in the [workbook](../exercises/01_internals_typing_architecture_exercises.ipynb).

In [ ]:
from lesson import Exercises, ___, run

check = Exercises('advanced/12', {1: '17a508f906205dad', 2: '3a0f9230d1fb0962', 3: '0d6e73180693e197', 4: '2187554323522ac4'})

**Exercise 1.** Keep the domain pure: pricing logic takes plain values and does no I/O.

In [ ]:
def final_price(base, discount_percent, tax_rate):
    discounted = base * (1 - discount_percent / 100)
    return round(discounted * (1 + ___), 2)

check(1, [final_price(100, 10, 0.2), final_price(50, 0, 0)])

**Exercise 2.** A small repository port with an in-memory adapter.

In [ ]:
from typing import Protocol

class Repo(Protocol):
    def get(self, key: str): ...
    def put(self, key: str, value) -> None: ...

class MemoryRepo:
    def __init__(self): self.data = {}
    def get(self, key): return self.data.get(key)
    def put(self, key, value): self.data[key] = ___

def rename(repo: Repo, key, new_name):
    item = repo.get(key)
    repo.put(key, {**item, "name": new_name})

r = MemoryRepo(); r.put("u1", {"name": "old"})
rename(r, "u1", "new")
check(2, r.get("u1"))

**Exercise 3.** Inject the clock so expiry can be tested without waiting.

In [ ]:
from datetime import datetime, timedelta

def is_expired(created, ttl, now):
    return now() >= created + ___

start = datetime(2024, 1, 1)
check(3, [is_expired(start, timedelta(hours=1), lambda: start + timedelta(minutes=59)),
       is_expired(start, timedelta(hours=1), lambda: start + timedelta(hours=1))])

**Exercise 4.** A domain-specific error keeps boundaries clear.

In [ ]:
class OutOfStock(Exception):
    pass

def reserve(stock, item, qty):
    if stock.get(item, 0) < qty:
        raise ___(item)
    stock[item] -= qty
    return stock[item]

stock = {"pen": 3}
try:
    reserve(stock, "pen", 5)
    outcome = "reserved"
except OutOfStock as error:
    outcome = f"out of stock: {error}"
check(4, (outcome, reserve(stock, "pen", 2)))

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** In ports and adapters, the domain code should…

   - **a)** Depend on small interfaces (ports), not concrete I/O
   - **b)** Call the database directly
   - **c)** Read environment variables everywhere

**2.** Injecting a clock lets you…

   - **a)** Test time-dependent logic deterministically
   - **b)** Avoid datetime
   - **c)** Make code faster

**3.** A fake repository in tests is…

   - **a)** Production code
   - **b)** A simple in-memory implementation of the port
   - **c)** A mocked database server

In [ ]:
from quiz import Quiz

quiz = Quiz('advanced/12', {1: '73bb11f6f67a', 2: '99925a6649ad', 3: '944ad110d95b'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [13 — Security: Validation, Safe Parsing, Secrets and Dependencies](13_security_safe_parsing_secrets.ipynb)